# G-Generation
Graph-grounded answer generation using KuzuDB subgraph context + GPT-OSS-20b via HuggingFace Inference API (LangChain)

In [1]:
!pip install -q transformers sentencepiece kuzu pyvis sentence-transformers
!pip install -q langchain langchain-huggingface huggingface_hub langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.9 MB/s eta 0:00:00


### Imports

In [2]:
import os
import re
import csv
import json
import uuid
import shutil
from pathlib import Path

import torch
import kuzu
import numpy as np
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

### Config

In [3]:
import os
from google.colab import userdata

# Retrieve the secret from Colab
hf_token = userdata.get('HUGGINGFACEHUB_API_TOKEN')

# Set it as an environment variable for LangChain
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

print("Secret loaded successfully!")

Secret loaded successfully!


In [4]:
KUZU_DB_PATH    = "./complete_dataset_final_kg.kuzu"
HF_MODEL_ID     = "openai/gpt-oss-120b"                 # GPT-OSS-20b on HF Hub — update if slug differs
TOP_K           = 5
HOPS            = 1

### Connection to DB

In [5]:
db   = kuzu.Database(KUZU_DB_PATH)
conn = kuzu.Connection(db)

### Embedding Model

In [6]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed_query(query: str):
    return embedder.encode(query, normalize_embeddings=True).tolist()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Getting the Seed Node

In [7]:
def get_seed_nodes(query: str, conn, top_k: int = TOP_K):
    q_vec = embedder.encode(query, normalize_embeddings=True).tolist()

    result = conn.execute("""
        CALL QUERY_VECTOR_INDEX('Entity', 'entity_hnsw', $query_vec, $top_k)
        YIELD node, distance
        RETURN node.entity_id, node.name, distance
        ORDER BY distance ASC
    """, {"query_vec": q_vec, "top_k": top_k})

    return result.get_as_df().rename(columns={
        "node.entity_id": "entity_id",
        "node.name":      "name",
    })

### Get the Subgraph

In [8]:
def get_subgraph(seed_entity_ids: list, conn, hops: int = HOPS):
    id_list = "[" + ", ".join(f'"{eid}"' for eid in seed_entity_ids) + "]"

    query = f"""
        MATCH (h:Entity)-[r:Graph*1..{hops}]->(t:Entity)
        WHERE h.entity_id IN {id_list}
        UNWIND relationships(r) AS rel
        RETURN DISTINCT
            h.name              AS head,
            rel.relation        AS relation,
            t.name              AS tail,
            rel.source_file     AS source,
            rel.page_number     AS page
    """
    result = conn.execute(query)
    return result.get_as_df()

### Flatten the Graph

In [9]:
def subgraph_to_context(subgraph_df) -> str:
    """
    Converts a subgraph DataFrame into a clean, LLM-readable context string.
    - Deduplicates identical (head, relation, tail) triples
    - Groups triples by source file
    - Within each group, sorts by head entity then relation
    """
    if subgraph_df.empty:
        return "No relevant knowledge graph context found."

    # 1. Deduplicate on the triple itself; keep first occurrence (preserves source/page)
    deduped = (
        subgraph_df
        .drop_duplicates(subset=["head", "relation", "tail"])
        .sort_values(["source", "page", "head", "relation"])
    )

    # 2. Group by source file and build sections
    sections = []
    for source, group in deduped.groupby("source", sort=True):
        header = f"[Source: {source}]"
        triples = []
        for _, row in group.iterrows():
            triples.append(
                f"  - {row['head']} --[{row['relation']}]--> {row['tail']}"
                f"  (p.{row['page']})"
            )
        sections.append(header + "\n" + "\n".join(triples))

    total = len(deduped)
    original = len(subgraph_df)
    removed = original - total
    summary = f"Knowledge Graph Context — {total} triples"
    if removed:
        summary += f" ({removed} duplicates removed)"

    return summary + "\n\n" + "\n\n".join(sections)

### LLM Setup — GPT-OSS-20b via HuggingFace Inference API

In [10]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
llm_endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="conversational",
    max_new_tokens=2048,
)
llm = ChatHuggingFace(llm=llm_endpoint)

### Prompt Template

In [11]:
GENERATION_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a knowledgeable assistant. Use ONLY the knowledge graph triples below to answer the question.
Each triple is in the form: (head) -[relation]-> (tail)  [src: <file>, p.<page>]
If the triples do not contain enough information, say so clearly.

Knowledge Graph Context:
{context}

Question: {question}

Answer:"""
)

chain = LLMChain(llm=llm, prompt=GENERATION_PROMPT)

/tmp/ipykernel_3116/1961200801.py:15: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=GENERATION_PROMPT)


### G-Generation

In [12]:
def g_generate(query: str, conn, top_k: int = TOP_K, hops: int = HOPS) -> str:
    # --- Retrieval (same as G-Retrieval) ---
    seeds = get_seed_nodes(query, conn, top_k=top_k)
    print("Seed nodes:\n", seeds)

    seed_ids = seeds["entity_id"].tolist()
    subgraph = get_subgraph(seed_ids, conn, hops=hops)
    print(f"\nSubgraph size: {len(subgraph)} triples")

    context = subgraph_to_context(subgraph)
    print(f"\nContext size: {len(context)}")
    print(context)
    # --- Generation ---
    response = chain.invoke({"context": context, "question": query})
    answer = response["text"].strip()
    return answer

# Try it
answer = g_generate("How do functional dependencies lead to normalization, and how does the resulting schema affect index design?", conn, TOP_K, 2)
print("\n===== Generated Answer =====")
print(answer)

Seed nodes:
                               entity_id  \
0  a4d20d9b-9569-47e1-827b-4a7624266d43   
1  a439a921-8f15-445c-8231-74e4fe5042ef   
2  0a38a271-f415-46d4-903c-0cb87d81257a   
3  78e0249f-99b2-4415-b8e5-e1c3e826d6cf   
4  bbf4e827-5f63-4fc8-a145-f8875044ed21   

                                              name  distance  
0                           functional-based index  0.454612  
1        further topics in functional dependencies  0.459487  
2  database design theory and normalization part 6  0.483189  
3                                normalized design  0.484797  
4                     full functional dependencies  0.486212  

Subgraph size: 92 triples

Context size: 5752
Knowledge Graph Context — 67 triples (25 duplicates removed)

[Source: adbms.pdf]
  - database design theory and normalization part 6 --[subclass of]--> data model  (p.88)
  - database design theory and normalization part 6 --[part of]--> database design  (p.94)
  - further topics in functional depende

### Debug — Inspect Subgraph Directly

In [13]:
seeds    = get_seed_nodes("normalization denormalization", conn)
seed_ids = seeds["entity_id"].tolist()
subgraph = get_subgraph(seed_ids, conn, hops=2)
print(subgraph)

                             head      relation                         tail  \
0    normalization transformation   subclass of            similarity search   
1                 denormalization   subclass of               design process   
2                 denormalization       follows            conceptual design   
3                 denormalization   subclass of              database design   
4                 denormalization   subclass of              database design   
..                            ...           ...                          ...   
466                 normalization  practiced by                          dba   
467                 normalization           use                  statistical   
468                 normalization   subclass of  representative transactions   
469                 normalization   subclass of  representative transactions   
470                 normalization      facet of                     metadata   

        source  page  
0      dwm.pdf  

In [14]:
seed_ids = seeds["entity_id"].tolist()
id_list  = "[" + ", ".join(f'"{eid}"' for eid in seed_ids) + "]"

result = conn.execute(f"""
    MATCH (h:Entity)-[r:Graph]->(t:Entity)
    WHERE h.entity_id IN {id_list}
    RETURN h.name, r.relation, t.name
""")
print(result.get_as_df())

                          h.name   r.relation  \
0                denormalization  opposite of   
1                  normalization      part of   
2                  normalization     facet of   
3                  normalization  subclass of   
4                  normalization  subclass of   
5                  normalization  subclass of   
6                  normalization  subclass of   
7                  normalization  subclass of   
8                  normalization  subclass of   
9                denormalization  subclass of   
10               denormalization  subclass of   
11                 normalization      part of   
12                 normalization      part of   
13  normalization transformation  subclass of   
14                 normalization  subclass of   
15                 normalization  opposite of   
16                 normalization  subclass of   
17                 normalization  subclass of   
18                 normalization  subclass of   

                   

In [15]:
node_count = conn.execute("MATCH (e:Entity) RETURN count(e)").get_next()[0]
rel_count  = conn.execute("MATCH ()-[r:Graph]->()  RETURN count(r)").get_next()[0]
print(f"Entity nodes : {node_count}")
print(f"Graph edges  : {rel_count}")

print("\nSample (50 edges):")
sample = conn.execute("""
    MATCH (h:Entity)-[r:Graph]->(t:Entity)
    RETURN h.name, r.source_file, r.page_number, r.relation, t.name
    LIMIT 50
""")
print(f"{'HEAD':<25}  {'SOURCE':<20}  {'PAGE':>4}  {'RELATION':<25}  TAIL")
print("-" * 100)
for row in sample:
    print(f"{row[0][:23]:<25}  {row[1][:18]:<20}  {row[2]:>4}  {row[3][:23]:<25}  {row[4]}")

Entity nodes : 15340
Graph edges  : 29776

Sample (50 edges):
HEAD                       SOURCE                PAGE  RELATION                   TAIL
----------------------------------------------------------------------------------------------------
sim(s1,s4)                 bda.pdf                103  instance of                signature matrix
ethernet                   dbms.pdf               620  uses                       ip
ethernet                   adbms.pdf              652  uses                       ip
ethernet                   adbms.pdf              652  uses                       ip
laptop                     dwm.pdf                 53  subclass of                computer
laptop                     dwm.pdf                534  subclass of                computer
bellaire, tx               dbms.pdf               220  located in the administ    bellaire, tx
bellaire, tx               dbms.pdf               272  located in the administ    houston, tx
bellaire, tx             